In [1]:
#============================================
# Celda 1 — Imports y carga del JSON
#============================================
import json, glob, pandas as pd, numpy as np, os
from pathlib import Path
from datetime import datetime

def ultimo_json(carpeta_base: str) -> str:
    """Devuelve el JSON más reciente por fecha real DD_MM_YYYY.json"""
    archivos = glob.glob(f'{carpeta_base}/**/*.json', recursive=True)
    assert len(archivos) > 0, f"❌ No hay JSONs en {carpeta_base}"
    def parse_fecha(ruta):
        try:
            return datetime.strptime(Path(ruta).stem, "%d_%m_%Y")
        except ValueError:
            return datetime.min
    return max(archivos, key=parse_fecha)

# ── ambiental ──
f = ultimo_json('../../data/ambiental')
data = json.load(open(f))
print(f'Archivo cargado: {f}')
print(f'Keys disponibles: {list(data.keys())}')


Archivo cargado: ../../data/ambiental/2026/04/14_04_2026.json
Keys disponibles: ['timestamp', 'observacion_meteorologica', 'calidad_aire']


In [2]:
#============================================
# Celda 2 — Explorar estructura AEMET
#============================================
aemet_raw  = data['observacion_meteorologica']
estaciones = aemet_raw['estaciones']

print(f'Total estaciones: {aemet_raw["total_estaciones"]}')
print(f'Timestamp captura: {aemet_raw["timestamp_captura"]}')
print(f'\nKeys de una estación:')
print(list(estaciones[0].keys()))
print(f'\nEjemplo estación[0]:')
print(json.dumps(estaciones[0], indent=2, ensure_ascii=False))


Total estaciones: 10137
Timestamp captura: 2026-04-14T13:48:15.449416

Keys de una estación:
['idema', 'ubi', 'lat', 'lon', 'alt', 'fint', 'ta', 'tamin', 'tamax', 'prec', 'hr', 'vv', 'vmax', 'dv', 'dmax', 'pres', 'pres_nmar', 'vis', 'inso', 'ts', 'tpr']

Ejemplo estación[0]:
{
  "idema": "0009X",
  "ubi": "ALFORJA",
  "lat": 41.213892,
  "lon": 0.963335,
  "alt": 406.0,
  "fint": "2026-04-14T01:00:00+0000",
  "ta": 7.3,
  "tamin": 6.7,
  "tamax": 7.9,
  "prec": 0.0,
  "hr": 68.0,
  "vv": 5.4,
  "vmax": 9.5,
  "dv": 290.0,
  "dmax": 272.0,
  "pres": null,
  "pres_nmar": null,
  "vis": null,
  "inso": null,
  "ts": null,
  "tpr": null
}


In [3]:
#============================================
# Celda 3 — Parsear AEMET a DataFrame
#============================================
df_aemet = pd.DataFrame(estaciones)

# Convertir timestamp
df_aemet['fint'] = pd.to_datetime(df_aemet['fint'], utc=True, errors='coerce')
df_aemet['fecha'] = df_aemet['fint'].dt.date

# Columnas numéricas
cols_num = ['lat','lon','alt','ta','tamin','tamax','prec','hr',
            'vv','vmax','dv','dmax','pres','pres_nmar','vis','inso','ts','tpr']
for c in cols_num:
    if c in df_aemet.columns:
        df_aemet[c] = pd.to_numeric(df_aemet[c], errors='coerce')

print(f'Shape: {df_aemet.shape}')
print(f'Columnas: {list(df_aemet.columns)}')
df_aemet.head()


Shape: (10137, 22)
Columnas: ['idema', 'ubi', 'lat', 'lon', 'alt', 'fint', 'ta', 'tamin', 'tamax', 'prec', 'hr', 'vv', 'vmax', 'dv', 'dmax', 'pres', 'pres_nmar', 'vis', 'inso', 'ts', 'tpr', 'fecha']


,idema,ubi,lat,lon,alt,fint,ta,tamin,tamax,prec,...,vmax,dv,dmax,pres,pres_nmar,vis,inso,ts,tpr,fecha
0,0009X,ALFORJA,41.213892,0.963335,406.0,2026-04-14 01:00:00+00:00,7.3,6.7,7.9,0.0,...,9.5,290.0,272.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-14
1,0016A,REUS AEROPUERTO,41.145000,1.163611,71.0,2026-04-14 01:00:00+00:00,11.1,11.0,11.7,0.0,...,9.8,280.0,290.0,1005.5,1014.7,30.0,0.0,9.9,2.7,2026-04-14
2,0034X,VALLS,41.293053,1.260838,233.0,2026-04-14 01:00:00+00:00,9.4,8.2,9.4,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-14
3,0042Y,TARRAGONA FAC. GEOGRAFIA,41.123892,1.249167,55.0,2026-04-14 01:00:00+00:00,9.0,9.0,9.6,0.0,...,2.1,56.0,252.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-14
4,0061X,PONTONS,41.417052,1.519269,632.0,2026-04-14 01:00:00+00:00,3.4,3.4,3.8,0.0,...,1.3,332.0,312.0,NaN,NaN,NaN,NaN,NaN,NaN,2026-04-14


In [4]:
#============================================
# Celda 4 — Análisis rápido AEMET
#============================================
print('Valores nulos por columna:')
nulos = df_aemet.isnull().sum()
print(nulos[nulos > 0])

print(f'\nRango temperatura:  {df_aemet["ta"].min():.1f}°C — {df_aemet["ta"].max():.1f}°C')
print(f'Media temperatura:  {df_aemet["ta"].mean():.1f}°C')
print(f'\nEstaciones con precipitación > 0: {(df_aemet["prec"] > 0).sum()}')
print(f'Precipitación máxima: {df_aemet["prec"].max():.1f} mm')
print(f'\nRango altitud: {df_aemet["alt"].min():.0f}m — {df_aemet["alt"].max():.0f}m')
print(f'\nEstaciones sin coordenadas: {df_aemet[["lat","lon"]].isnull().any(axis=1).sum()}')


Valores nulos por columna:
ta            150
tamin         150
tamax         150
prec          272
hr            152
vv           1386
vmax         1423
dv           1347
dmax         1396
pres         7241
pres_nmar    8305
vis          8894
inso         8422
ts           8499
tpr          7196
dtype: int64

Rango temperatura:  -9.4°C — 25.5°C
Media temperatura:  10.8°C

Estaciones con precipitación > 0: 364
Precipitación máxima: 42.2 mm

Rango altitud: 0m — 2856m

Estaciones sin coordenadas: 0


In [5]:
#============================================
# Celda 5 — Explorar estructura WAQI
#============================================
waqi_raw = data['calidad_aire']
ciudades = waqi_raw['ciudades']

print(f'Total ciudades OK:  {waqi_raw["total_ciudades_ok"]}')
print(f'Total errores:      {waqi_raw["total_errores"]}')
print(f'\nKeys de una ciudad:')
print(list(ciudades[0].keys()))
print(f'\nEjemplo ciudad[0]:')
print(json.dumps(ciudades[0], indent=2, ensure_ascii=False))


Total ciudades OK:  30
Total errores:      3

Keys de una ciudad:
['ciudad', 'nombre', 'lat', 'lon', 'timestamp', 'aqi', 'dominante', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO', 'temperatura', 'humedad', 'presion', 'viento']

Ejemplo ciudad[0]:
{
  "ciudad": "madrid",
  "nombre": "Madrid",
  "lat": 40.4167754,
  "lon": -3.7037902,
  "timestamp": "2026-02-09 14:00:00",
  "aqi": 28,
  "dominante": "o3",
  "NO2": 10.1,
  "PM25": 21,
  "PM10": 13,
  "O3": 28.1,
  "SO2": 0.6,
  "CO": 0.1,
  "temperatura": 10.5,
  "humedad": 99,
  "presion": 1009,
  "viento": 10
}


In [6]:
#============================================
# Celda 6 — Parsear WAQI a DataFrame
#============================================
df_waqi = pd.DataFrame(ciudades)

# Convertir tipos
df_waqi['timestamp'] = pd.to_datetime(df_waqi['timestamp'], errors='coerce')

cols_num = ['lat','lon','aqi','NO2','PM25','PM10','O3','SO2','CO',
            'temperatura','humedad','presion','viento']
for c in cols_num:
    if c in df_waqi.columns:
        df_waqi[c] = pd.to_numeric(df_waqi[c], errors='coerce')

print(f'Shape: {df_waqi.shape}')
print(f'Columnas: {list(df_waqi.columns)}')
df_waqi.head(10)


Shape: (30, 17)
Columnas: ['ciudad', 'nombre', 'lat', 'lon', 'timestamp', 'aqi', 'dominante', 'NO2', 'PM25', 'PM10', 'O3', 'SO2', 'CO', 'temperatura', 'humedad', 'presion', 'viento']


,ciudad,nombre,lat,lon,timestamp,aqi,dominante,NO2,PM25,PM10,O3,SO2,CO,temperatura,humedad,presion,viento
0,madrid,Madrid,40.416775,-3.703790,2026-02-09 14:00:00,28,o3,10.1,21.0,13.0,28.1,0.6,0.1,10.5,99.0,1009.0,10.0
1,barcelona,"Barcelona (Eixample), Catalunya, Spain",41.385343,2.153822,2026-04-14 15:00:00,20,o3,14.2,21.0,12.0,20.4,2.6,0.1,16.3,61.5,1018.6,4.5
2,valencia,"Pista de Silla, València, Valencia, Spain",39.456111,-0.375833,2026-02-13 02:00:00,34,pm25,1.9,34.0,12.0,29.3,1.6,0.1,13.8,63.0,NaN,0.6
3,sevilla,"Ranilla, Sevilla, Spain",37.384250,-5.959620,2026-04-14 15:00:00,27,pm25,4.6,27.0,12.0,21.0,1.6,0.1,21.3,56.2,1020.9,2.2
4,bilbao,"Mazarredo, Bilbao, País Vasco, Spain",43.267500,-2.935200,2026-04-14 13:00:00,9,pm25,3.7,9.0,5.0,NaN,2.6,0.1,18.0,65.5,1021.5,4.6
5,zaragoza,"El Picarral, Zaragoza, Spain",41.670278,-0.871111,2026-04-14 14:00:00,36,o3,2.1,12.0,NaN,36.4,NaN,1.6,20.0,35.0,1018.0,16.0
6,malaga,"Carranque, Malaga, Spain",36.719640,-4.447500,2026-04-14 15:00:00,29,o3,3.7,24.0,9.0,29.4,1.6,0.1,23.0,38.0,1020.1,8.8
7,murcia,"San Basilio Murcia Ciudad, Murcia, Spain",37.993960,-1.144628,2026-04-14 15:00:00,34,o3,5.7,10.0,8.0,34.2,1.8,0.1,22.7,37.3,1017.4,7.8
8,palma,"foners, Baleares, Spain",39.571250,2.657028,2026-02-09 14:00:00,32,o3,5.1,30.0,15.0,31.7,1.9,0.1,16.1,56.0,1002.7,4.0
9,alicante,"Rabassa, Alacant, Valencia, Spain",38.351111,-0.513889,2026-02-13 04:00:00,29,o3,2.3,9.0,3.0,28.5,1.6,0.1,15.6,10.0,1021.7,2.4


In [7]:
#============================================
# Celda 7 — Análisis rápido WAQI
#============================================
def nivel_aqi(aqi):
    if pd.isna(aqi):       return 'Sin datos'
    elif aqi <= 50:        return '🟢 Bueno'
    elif aqi <= 100:       return '🟡 Moderado'
    elif aqi <= 150:       return '🟠 Insalubre sensibles'
    elif aqi <= 200:       return '🔴 Insalubre'
    else:                  return '🟣 Muy insalubre'

df_waqi['nivel_aqi'] = df_waqi['aqi'].apply(nivel_aqi)

print('Ciudades por nivel AQI:')
print(df_waqi['nivel_aqi'].value_counts().to_string())

print(f'\nTop 5 peor AQI:')
print(df_waqi.nlargest(5, 'aqi')[['nombre','aqi','dominante','PM25','PM10','O3']].to_string(index=False))

print(f'\nTop 5 mejor AQI:')
print(df_waqi.nsmallest(5, 'aqi')[['nombre','aqi','dominante','PM25','PM10','O3']].to_string(index=False))

print(f'\nValores nulos por columna:')
print(df_waqi.isnull().sum()[df_waqi.isnull().sum() > 0])


Ciudades por nivel AQI:
nivel_aqi
🟢 Bueno       29
🟡 Moderado     1

Top 5 peor AQI:
                            nombre  aqi dominante  PM25  PM10   O3
Lope de Vega, Vigo, Galicia, Spain   57      pm25  57.0  21.0 19.2
                   Burgos I, Spain   45      pm10   9.0  45.0 16.3
                   Erie, Ohio, USA   39      pm25  39.0  14.0 19.2
     Constitución, Asturias, Spain   38      pm25  38.0  20.0 12.6
        Tome Cano, Canarias, Spain   37        o3  25.0  13.0 37.0

Top 5 mejor AQI:
                                     nombre  aqi dominante  PM25  PM10    O3
  Vila Velha - IBES, Espírito Santo, Brazil    4        o3   NaN   NaN  3.71
       Mazarredo, Bilbao, País Vasco, Spain    9      pm25   9.0   5.0   NaN
    Tarragona (Bonavista), Catalunya, Spain   13      pm25  13.0   8.0 13.90
Girona (Escola de Música), Catalunya, Spain   13      pm25  13.0   8.0 16.00
     Arco de ladrillo II, Valladolid, Spain   17      pm25  17.0   8.0 23.60

Valores nulos por columna:
NO2  

In [8]:
#============================================
# Celda 8 — Resumen calidad de datos
#============================================
resumen = pd.DataFrame([
    {
        'tabla':     'aemet_estaciones',
        'filas':     len(df_aemet),
        'columnas':  len(df_aemet.columns),
        'periodos':  df_aemet['fint'].nunique(),
        'nulos_%':   round(df_aemet.isnull().sum().sum() / (df_aemet.shape[0] * df_aemet.shape[1]) * 100, 1)
    },
    {
        'tabla':     'waqi_ciudades',
        'filas':     len(df_waqi),
        'columnas':  len(df_waqi.columns),
        'periodos':  df_waqi['timestamp'].nunique(),
        'nulos_%':   round(df_waqi.isnull().sum().sum()  / (df_waqi.shape[0]  * df_waqi.shape[1])  * 100, 1)
    },
])
print(resumen.to_string(index=False))


           tabla  filas  columnas  periodos  nulos_%
aemet_estaciones  10137        22        13     24.7
   waqi_ciudades     30        18        11      3.5


In [9]:
#============================================
# Celda 9 — Guardar parquets
#============================================
os.makedirs('../../output/ambiental/01-raw', exist_ok=True)

df_aemet.to_parquet('../../output/ambiental/01-raw/raw_aemet_estaciones.parquet', index=False)
print(f'✅ Guardado: output/ambiental/01-raw/raw_aemet_estaciones.parquet ({len(df_aemet)} filas)')

df_waqi.to_parquet('../../output/ambiental/raw_waqi_ciudades.parquet', index=False)
print(f'✅ Guardado: output/ambiental/raw_waqi_ciudades.parquet ({len(df_waqi)} filas)')

print('\n🎉 Todos los parquets ambiental guardados en output/')


✅ Guardado: output/ambiental/01-raw/raw_aemet_estaciones.parquet (10137 filas)
✅ Guardado: output/ambiental/raw_waqi_ciudades.parquet (30 filas)

🎉 Todos los parquets ambiental guardados en output/
